# Calibration des probabilités de défaut

Ce notebook suit les étapes suivantes :
1. Configuration & imports
2. Diagnostic de la calibration initiale
3. Platt Scaling (calibration)
4. Correction de l'intercept sur le taux de défaut cible
5. Validation post-calibration
6. Synthèse

## 0. Configuration — à renseigner

In [ ]:
# ── Noms des colonnes ──────────────────────────────────────────────────────
COL_CIBLE  = "defaut"        # colonne binaire (0/1)
COL_PROBA  = "score_proba"   # probabilité brute sortie du modèle
COL_DATE   = "date"          # colonne date ou période (pour PSI)

# ── Paramètres ────────────────────────────────────────────────────────────
N_DECILES  = 10              # nombre de groupes pour la courbe de calibration
ALPHA      = 0.05            # seuil de significativité du test Hosmer-Lemeshow
N_GROUPES_HL = 10            # nombre de groupes pour le test Hosmer-Lemeshow

## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.linear_model    import LogisticRegression
from sklearn.calibration     import calibration_curve
from sklearn.metrics         import brier_score_loss, roc_auc_score
from scipy                   import stats

# Style graphique
plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.spines.top":  False,
    "axes.spines.right":False,
    "axes.grid":        True,
    "grid.alpha":       0.3,
})

print("Imports OK")

## 2. Chargement des données

> **À adapter** : remplacer par ton chargement réel de `df_train` et `df_val`.

In [ ]:
# ── Charger tes DataFrames ici ────────────────────────────────────────────
# df_train = pd.read_csv("...")
# df_val   = pd.read_csv("...")

# Vérification rapide
print(f"Train : {len(df_train):,} lignes | taux défaut = {df_train[COL_CIBLE].mean():.2%}")
print(f"Val   : {len(df_val):,} lignes | taux défaut = {df_val[COL_CIBLE].mean():.2%}")

# Le taux de défaut cible = taux observé sur la validation
TAUX_DEFAUT_CIBLE = df_val[COL_CIBLE].mean()
print(f"\nTaux de défaut cible (val) : {TAUX_DEFAUT_CIBLE:.4%}")

## 3. Fonctions utilitaires

In [ ]:
def hosmer_lemeshow(y_true: np.ndarray, y_prob: np.ndarray, n_groupes: int = 10) -> dict:
    """
    Test de Hosmer-Lemeshow.
    H0 : le modèle est bien calibré globalement.
    Un p-value < alpha → calibration insuffisante.
    """
    df = pd.DataFrame({"y": y_true, "p": y_prob})
    df["groupe"] = pd.qcut(df["p"], q=n_groupes, duplicates="drop", labels=False)

    stats_g = df.groupby("groupe").agg(
        n      =("y", "count"),
        obs_1  =("y", "sum"),
        pred_p =("p", "mean"),
    )
    stats_g["obs_0"]  = stats_g["n"] - stats_g["obs_1"]
    stats_g["pred_1"] = stats_g["n"] * stats_g["pred_p"]
    stats_g["pred_0"] = stats_g["n"] - stats_g["pred_1"]

    # Statistique Chi-2 de HL
    hl_stat = (
        (stats_g["obs_1"] - stats_g["pred_1"])**2 / stats_g["pred_1"].clip(lower=1e-10) +
        (stats_g["obs_0"] - stats_g["pred_0"])**2 / stats_g["pred_0"].clip(lower=1e-10)
    ).sum()

    ddl   = n_groupes - 2
    p_val = 1 - stats.chi2.cdf(hl_stat, df=ddl)

    return {"hl_stat": round(hl_stat, 3), "ddl": ddl, "p_value": round(p_val, 4)}


def métriques_calibration(y_true: np.ndarray, y_prob: np.ndarray, label: str = "") -> dict:
    """Calcule les métriques clés de calibration et discrimination."""
    hl  = hosmer_lemeshow(y_true, y_prob, N_GROUPES_HL)
    auc = roc_auc_score(y_true, y_prob)
    bs  = brier_score_loss(y_true, y_prob)
    pd_moy = y_prob.mean()
    tx_def = y_true.mean()

    résultat = {
        "label":       label,
        "AUC":         round(auc, 4),
        "Gini":        round(2 * auc - 1, 4),
        "Brier Score": round(bs, 4),
        "PD moyenne":  round(pd_moy, 4),
        "Taux défaut": round(tx_def, 4),
        "Écart PD-TD": round(pd_moy - tx_def, 4),
        "HL stat":     hl["hl_stat"],
        "HL p-value":  hl["p_value"],
        "HL résultat": "✅ Bien calibré" if hl["p_value"] >= ALPHA else "❌ Mal calibré",
    }
    return résultat


def tracer_calibration(y_true, y_prob, titre: str = "", ax=None, n_bins: int = 10):
    """
    Trace la courbe de calibration (reliability diagram).
    Chaque point = un décile : PD moyenne prédite vs taux défaut observé.
    La diagonale = calibration parfaite.
    """
    prob_true, prob_pred = calibration_curve(y_true, y_prob, n_bins=n_bins, strategy="quantile")

    if ax is None:
        fig, ax = plt.subplots(figsize=(6, 6))

    ax.plot([0, 1], [0, 1], "--", color="grey", label="Calibration parfaite", linewidth=1.5)
    ax.plot(prob_pred, prob_true, "o-", color="#1f77b4", label="Modèle", linewidth=2, markersize=7)

    # Annotation des écarts
    for xp, yp in zip(prob_pred, prob_true):
        ax.annotate(
            f"{yp - xp:+.3f}",
            xy=(xp, yp), xytext=(5, 3), textcoords="offset points",
            fontsize=7, color="#d62728" if yp < xp else "#2ca02c"
        )

    ax.set_xlabel("PD moyenne prédite", fontsize=11)
    ax.set_ylabel("Taux défaut observé", fontsize=11)
    ax.set_title(titre, fontsize=12, fontweight="bold")
    ax.legend(fontsize=9)
    ax.set_xlim(0, max(prob_pred) * 1.15)
    ax.set_ylim(0, max(prob_true) * 1.15)

    return ax


def tracer_distribution_scores(y_true, y_prob, titre: str = "", ax=None):
    """
    Trace la distribution des scores prédits séparément pour
    défauts (1) et non-défauts (0).
    Permet de visualiser la discrimination et le positionnement des PD.
    """
    if ax is None:
        fig, ax = plt.subplots(figsize=(7, 4))

    ax.hist(y_prob[y_true == 0], bins=50, alpha=0.6, color="#2ca02c", label="Non-défaut (0)", density=True)
    ax.hist(y_prob[y_true == 1], bins=50, alpha=0.6, color="#d62728", label="Défaut (1)",     density=True)
    ax.axvline(y_prob[y_true == 0].mean(), color="#2ca02c", linestyle="--", linewidth=1.5)
    ax.axvline(y_prob[y_true == 1].mean(), color="#d62728", linestyle="--", linewidth=1.5)
    ax.set_xlabel("Score prédit", fontsize=11)
    ax.set_ylabel("Densité", fontsize=11)
    ax.set_title(titre, fontsize=12, fontweight="bold")
    ax.legend(fontsize=9)

    return ax


print("Fonctions définies OK")

## 4. Étape 1 — Diagnostic de la calibration initiale

On évalue la calibration **avant toute correction** sur le jeu de validation.
Objectif : quantifier l'écart entre ce que le modèle prédit et ce qui est observé.

In [ ]:
y_val   = df_val[COL_CIBLE].values
p_val   = df_val[COL_PROBA].values

# ── Métriques initiales ───────────────────────────────────────────────────
métriques_init = métriques_calibration(y_val, p_val, label="Avant calibration")
print("=" * 55)
print("DIAGNOSTIC INITIAL — Jeu de validation")
print("=" * 55)
for k, v in métriques_init.items():
    print(f"  {k:<20} {v}")

In [ ]:
# ── Graphiques de diagnostic initial ─────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle("Diagnostic de calibration — Avant correction", fontsize=14, fontweight="bold", y=1.01)

tracer_calibration(
    y_val, p_val,
    titre="Courbe de calibration (reliability diagram)",
    ax=axes[0], n_bins=N_DECILES
)
tracer_distribution_scores(
    y_val, p_val,
    titre="Distribution des scores prédits",
    ax=axes[1]
)

plt.tight_layout()
plt.show()

# ── Interprétation automatique ────────────────────────────────────────────
écart = métriques_init["Écart PD-TD"]
if abs(écart) < 0.001:
    print("✅ PD moyenne très proche du taux observé — calibration globale satisfaisante.")
elif écart > 0:
    print(f"🔴 Sur-estimation : le modèle prédit en moyenne {écart:.2%} de plus que le taux observé.")
else:
    print(f"🟡 Sous-estimation : le modèle prédit en moyenne {abs(écart):.2%} de moins que le taux observé.")

## 5. Étape 2 — Platt Scaling

On ajuste une **régression logistique** (intercept + coefficient) sur les scores bruts
du jeu de **train**, pour apprendre la transformation `score → probabilité calibrée`.

La calibration est ensuite appliquée sur le jeu de **validation**.

In [ ]:
y_train = df_train[COL_CIBLE].values
p_train = df_train[COL_PROBA].values

# ── Ajustement du Platt Scaling sur le train ──────────────────────────────
# On utilise le score brut (logit ou proba) comme unique feature
# La régression logistique apprend A et B tels que :
#   P_calibrée = sigmoid(A * score + B)

platt = LogisticRegression(C=1e10)  # C très grand = pas de régularisation
platt.fit(p_train.reshape(-1, 1), y_train)

A = platt.coef_[0][0]
B = platt.intercept_[0]
print(f"Paramètres Platt Scaling : A = {A:.4f} | B = {B:.4f}")
print(f"  A > 1 → le modèle initial était trop confiant (scores trop extrêmes)")
print(f"  B < 0 → le modèle initial sur-estimait globalement")

# ── Application sur le jeu de validation ─────────────────────────────────
p_val_platt = platt.predict_proba(p_val.reshape(-1, 1))[:, 1]

# ── Métriques post-Platt ──────────────────────────────────────────────────
métriques_platt = métriques_calibration(y_val, p_val_platt, label="Après Platt Scaling")
print("\n" + "=" * 55)
print("APRÈS PLATT SCALING — Jeu de validation")
print("=" * 55)
for k, v in métriques_platt.items():
    print(f"  {k:<20} {v}")

In [ ]:
# ── Graphiques post-Platt ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle("Calibration après Platt Scaling", fontsize=14, fontweight="bold", y=1.01)

tracer_calibration(y_val, p_val_platt, titre="Courbe de calibration — Post Platt", ax=axes[0], n_bins=N_DECILES)
tracer_distribution_scores(y_val, p_val_platt, titre="Distribution des scores calibrés", ax=axes[1])

plt.tight_layout()
plt.show()

## 6. Étape 3 — Correction de l'intercept

Si la PD moyenne post-Platt ne correspond toujours pas au taux de défaut cible
(taux observé sur la validation), on ajuste l'intercept `B` pour forcer l'alignement.

$$B_{\text{corrigé}} = B + \ln\left(\frac{\pi_{\text{cible}}}{1-\pi_{\text{cible}}} \times \frac{1-\pi_{\text{train}}}{\pi_{\text{train}}}\right)$$

In [ ]:
# ── Taux de défaut sur train et cible ────────────────────────────────────
pi_train  = y_train.mean()
pi_cible  = TAUX_DEFAUT_CIBLE

# ── Correction de l'intercept ────────────────────────────────────────────
correction = np.log(
    (pi_cible / (1 - pi_cible)) * ((1 - pi_train) / pi_train)
)
B_corrigé  = B + correction

print(f"Taux défaut train  : {pi_train:.4%}")
print(f"Taux défaut cible  : {pi_cible:.4%}")
print(f"Correction log-odds : {correction:+.4f}")
print(f"B initial : {B:.4f} → B corrigé : {B_corrigé:.4f}")

# ── Application de la correction ─────────────────────────────────────────
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

# Score logit = A * score_brut + B_corrigé
logit_corrigé  = A * p_val + B_corrigé
p_val_corrigé  = sigmoid(logit_corrigé)

# ── Métriques post-correction ─────────────────────────────────────────────
métriques_corr = métriques_calibration(y_val, p_val_corrigé, label="Après correction intercept")
print("\n" + "=" * 55)
print("APRÈS CORRECTION INTERCEPT — Jeu de validation")
print("=" * 55)
for k, v in métriques_corr.items():
    print(f"  {k:<20} {v}")

In [ ]:
# ── Graphiques post-correction ────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle("Calibration après correction de l'intercept", fontsize=14, fontweight="bold", y=1.01)

tracer_calibration(y_val, p_val_corrigé, titre="Courbe de calibration — Post correction", ax=axes[0], n_bins=N_DECILES)
tracer_distribution_scores(y_val, p_val_corrigé, titre="Distribution des scores corrigés", ax=axes[1])

plt.tight_layout()
plt.show()

## 7. Étape 4 — Validation finale & comparaison des étapes

On compare côte à côte les trois versions du score :
- Score brut initial
- Après Platt Scaling
- Après correction de l'intercept

In [ ]:
# ── Tableau comparatif ────────────────────────────────────────────────────
df_comparaison = pd.DataFrame([
    métriques_init,
    métriques_platt,
    métriques_corr,
]).set_index("label")

print("\nCOMPARAISON DES ÉTAPES DE CALIBRATION")
print("=" * 80)
print(df_comparaison.to_string())

In [ ]:
# ── Graphique comparatif des courbes de calibration ───────────────────────
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
fig.suptitle("Comparaison des courbes de calibration", fontsize=14, fontweight="bold")

configs = [
    (p_val,          "Score brut"),
    (p_val_platt,    "Après Platt Scaling"),
    (p_val_corrigé,  "Après correction intercept"),
]

for ax, (proba, titre) in zip(axes, configs):
    tracer_calibration(y_val, proba, titre=titre, ax=ax, n_bins=N_DECILES)

plt.tight_layout()
plt.show()

In [ ]:
# ── Graphique : évolution des métriques clés ──────────────────────────────
étapes  = ["Brut", "Platt", "Corrigé"]
probas  = [p_val, p_val_platt, p_val_corrigé]

pd_moy  = [p.mean()                          for p in probas]
bs_vals = [brier_score_loss(y_val, p)        for p in probas]
hl_pval = [hosmer_lemeshow(y_val, p)["p_value"] for p in probas]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("Évolution des métriques de calibration", fontsize=13, fontweight="bold")

# PD moyenne vs taux cible
axes[0].bar(étapes, pd_moy, color="#1f77b4", alpha=0.8)
axes[0].axhline(pi_cible, color="red", linestyle="--", linewidth=1.5, label=f"Taux cible ({pi_cible:.2%})")
axes[0].set_title("PD moyenne prédite", fontweight="bold")
axes[0].set_ylabel("PD moyenne")
axes[0].legend(fontsize=8)
for i, v in enumerate(pd_moy):
    axes[0].text(i, v + 0.0002, f"{v:.4%}", ha="center", fontsize=9)

# Brier Score
axes[1].bar(étapes, bs_vals, color="#ff7f0e", alpha=0.8)
axes[1].set_title("Brier Score (↓ mieux)", fontweight="bold")
axes[1].set_ylabel("Brier Score")
for i, v in enumerate(bs_vals):
    axes[1].text(i, v + 0.0001, f"{v:.4f}", ha="center", fontsize=9)

# HL p-value
couleurs_hl = ["#2ca02c" if p >= ALPHA else "#d62728" for p in hl_pval]
axes[2].bar(étapes, hl_pval, color=couleurs_hl, alpha=0.8)
axes[2].axhline(ALPHA, color="black", linestyle="--", linewidth=1.5, label=f"Seuil α = {ALPHA}")
axes[2].set_title("HL p-value (↑ mieux)", fontweight="bold")
axes[2].set_ylabel("p-value")
axes[2].legend(fontsize=8)
for i, v in enumerate(hl_pval):
    axes[2].text(i, v + 0.005, f"{v:.3f}", ha="center", fontsize=9)

plt.tight_layout()
plt.show()

## 8. Étape 5 — PSI sur les PD calibrées

Le PSI mesure la **stabilité** de la distribution des PD calibrées entre train et validation.
Un PSI < 0.1 indique une distribution stable.
Un PSI > 0.25 indique un glissement significatif.

In [ ]:
def calculer_psi(p_ref: np.ndarray, p_test: np.ndarray, n_bins: int = 10) -> float:
    """
    Calcule le PSI entre deux distributions de probabilités.
    p_ref  : distribution de référence (train)
    p_test : distribution à comparer (validation)
    """
    # Seuils basés sur la distribution de référence
    seuils = np.quantile(p_ref, np.linspace(0, 1, n_bins + 1))
    seuils[0]  -= 1e-10
    seuils[-1] += 1e-10

    pct_ref  = np.histogram(p_ref,  bins=seuils)[0] / len(p_ref)
    pct_test = np.histogram(p_test, bins=seuils)[0] / len(p_test)

    eps = 1e-10
    psi = np.sum((pct_test - pct_ref) * np.log((pct_test + eps) / (pct_ref + eps)))
    return round(float(psi), 4)


# PD calibrées sur le train (pour comparaison)
logit_train_corrigé = A * p_train + B_corrigé
p_train_corrigé     = sigmoid(logit_train_corrigé)

psi_val = calculer_psi(p_train_corrigé, p_val_corrigé)

print(f"PSI (train vs val) sur PD calibrées : {psi_val}")
if psi_val < 0.10:
    print("✅ Distribution stable")
elif psi_val < 0.25:
    print("⚠️  Glissement modéré — à surveiller")
else:
    print("❌ Glissement significatif — recalibration nécessaire")

# ── Graphique : distribution PD train vs val ──────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(p_train_corrigé, bins=50, alpha=0.5, color="#1f77b4", label="Train",      density=True)
ax.hist(p_val_corrigé,   bins=50, alpha=0.5, color="#ff7f0e", label="Validation", density=True)
ax.axvline(p_train_corrigé.mean(), color="#1f77b4", linestyle="--", linewidth=1.5)
ax.axvline(p_val_corrigé.mean(),   color="#ff7f0e", linestyle="--", linewidth=1.5)
ax.set_xlabel("PD calibrée", fontsize=11)
ax.set_ylabel("Densité", fontsize=11)
ax.set_title(f"Distribution des PD calibrées — Train vs Validation (PSI = {psi_val})",
             fontsize=12, fontweight="bold")
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

## 9. Synthèse & export

Paramètres de calibration finale à conserver pour la mise en production.

In [ ]:
print("=" * 55)
print("SYNTHÈSE — Paramètres de calibration finale")
print("=" * 55)
print(f"  Méthode           : Platt Scaling + correction intercept")
print(f"  Coefficient A     : {A:.6f}")
print(f"  Intercept B       : {B:.6f}")
print(f"  Correction Δ      : {correction:+.6f}")
print(f"  Intercept corrigé : {B_corrigé:.6f}")
print(f"  Formule           : PD = sigmoid({A:.4f} × score_brut + {B_corrigé:.4f})")
print()
print("MÉTRIQUES FINALES (validation)")
print("=" * 55)
for k, v in métriques_corr.items():
    print(f"  {k:<20} {v}")

# ── Ajout des PD calibrées au DataFrame de validation ────────────────────
df_val["pd_calibrée"] = p_val_corrigé
df_train["pd_calibrée"] = p_train_corrigé

print("\n✅ Colonnes 'pd_calibrée' ajoutées à df_train et df_val.")
print(f"   Formule à appliquer en production :")
print(f"   pd_calibrée = 1 / (1 + exp(-({A:.4f} * score_brut + {B_corrigé:.4f})))")